<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Langchain(Prompt_Engineering_Part_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OUTPUT PARSERS

In [2]:
!pip install -q -U langchain langchain-google-genai python-dotenv

In [3]:
!pip install -U google-genai langchain-google-genai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.6
    Uninstalling langchain-core-1.6.6:
      Successfully uninstalled langchain-core-1.6.6


In [4]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in Colab Secrets.")

api_key = api_key.strip()

os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully.")
print("Key length:", len(api_key))
print("Key prefix:", api_key[:5])

API key loaded successfully.
Key length: 53
Key prefix: AQ.Ab


#1. StrOutputParser

This is the simplest parser.
Concept
Suppose Gemini returns an AIMessage:
AIMessage(
    content="Machine Learning is..."
)

But your application may only want:
Machine Learning is...

So:
Prompt
   ↓
Gemini
   ↓
AIMessage
   ↓
StrOutputParser
   ↓
String

***Basic Program***

from langchain_core.prompts import PromptTemplatefrom langchain_core.output_parsers import StrOutputParserprompt = PromptTemplate(    input_variables=["topic"],    template="Explain {topic} in simple language.")parser = StrOutputParser()


Now create the chain:
chain = prompt | llm | parser

In [5]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

In [6]:
from google import genai

client = genai.Client(
    api_key=api_key
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence."
)

print(response.text)

Hello there!


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

#model Evaluation
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.4
)

In [8]:
response=llm.invoke("Ishq kiya hai ? ")
print(response.content)

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 5h55m33.996133538s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTie

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple language."
)

parser = StrOutputParser()

In [ ]:
chain = prompt | llm | parser

In [ ]:
result = chain.invoke({
    "topic": "Machine Learning"
})

print(result)

# 2.StrOutputParser — Real Example

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert programming teacher."
    ),
    (
        "user",
        "Explain {topic} with a simple real-world example."
    )
])

chain = prompt | llm | StrOutputParser()

In [ ]:
result = chain.invoke({
    "topic": "RNN"
})

print(result)

In [ ]:
topics = [
    "Machine Learning",
    "RNN",
    "LSTM",
    "Transformer"
]

for topic in topics:

    result = chain.invoke({
        "topic": topic
    })

    print("\n====================")
    print(topic)
    print("====================")
    print(result)

# 3.JSON Output
Now we want Gemini to return data like:

{
    "name": "Rahul",
    "skills": ["Python", "SQL"],
    "experience": "3 years"
}

The important point:
JSON is a format.

We're telling Gemini:
Don't give me a paragraph. Give me data in this structure.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Extract information from the following text.

Return ONLY valid JSON.

Required fields:
- name
- skills
- experience

Text:
{text}

JSON:
"""
)

chain = prompt | llm | StrOutputParser()

In [ ]:
text = """
My name is Rahul Kumar.
I have 3 years of experience in Python.
I know Python, SQL, Machine Learning and TensorFlow.
"""

In [ ]:
result=chain.invoke({
    "text": text
})

print(result)


# 4.StructuredOutputParser
Now we move to a more LangChain-specific parser.
We want:

In [ ]:
from langchain.output_parsers import StructuredOutputParser
from langchain.output_parsers import ResponseSchema
from langchain_core.output_parsers import StructuredOutputParser, ResponseSchema

In [ ]:
#Define Fields
response_schemas = [
    ResponseSchema(
        name="name",
        description="The name of the person."
    ),
    ResponseSchema(
        name="skills",
        description="The skills of the person."
    ),
    ResponseSchema(
        name="experience",
        description="The experience of the person."
    ),
     ResponseSchema(
        name="salary",
        description="Expected or current salary"
    )
]

#Create parser:
parser = StructuredOutputParser.from_response_schemas(response_schemas)

#Get Formatting Instructions
format_instructions = parser.get_format_instructions()

print(format_instructions)



#Create Prompt
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
Extract information from the following text.

Text:
{text}

{format_instructions}
""",

    input_variables=["text"],

    partial_variables={
        "format_instructions": format_instructions
    }
)

chain = prompt | llm | parser

text = """
My name is Rahul Kumar.
I am 25 years old.
I know Python, SQL, Machine Learning and TensorFlow.
My expected salary is 8 LPA.
"""

result = chain.invoke({
    "text": text
})

print(result)

# 5.. Pydantic Structured Output

This is one of the most important programs in this phase.

Instead of simply saying:
name,
age,
skills

we define a Python model.

class Student(BaseModel):    name: str    age: int    skills: list[str]


Pydantic can validate that the returned data follows the expected structure and types.

# Step 1 — Create Pydantic Model

In [ ]:
from pydantic import BaseModel, Field

class Student(BaseModel):
    name: str
    age: int
    skills: list[str]

#Test the model:
student = Student(
    name="Rahul",
    age=24,
    skills=["Python", "SQL", "ML"]
)

print(student)

# Step 2 — Use Gemini Structured Output
With current LangChain model integrations, a convenient approach is to use the model's structured-output capability.

In [ ]:
structured_llm = llm.with_structured_output(Student)

# Step 3 — Create Prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Extract student information from the given text."
    ),
    (
        "user",
        "{text}"
    )
])

In [ ]:
chain = prompt | structured_llm

In [ ]:
text = """
My name is Rahul Kumar.
I am 24 years old.
I know Python, SQL, Machine Learning and TensorFlow.
"""

In [ ]:
result = chain.invoke({
    "text": text
})

print(result)

In [ ]:
print(type(result))
print(result.name)
print(result.age)
print(result.skills)